A DataFrame represents a rectangular table of data and contains an ordered, named collection of columns, each of which can be a different value types (numeric, string, Boolean, etc.,). The DataFrame has both a row and column index; it can be thought of as a dictionary of Series all sharing the same index. 



There are many ways to construct a DataFrame, though one of the most common is from a dictionary of equal-length lists or NumPy arrays:

In [846]:

import pandas as pd 
import numpy as np 


data = {"state":["Ohio", "Ohio", "Ohio", "Nevada", "Nevada", "Nevada"],
        "year":[2000, 2001, 2002, 2001, 2002, 2003],
        "pop": [1.5, 1.7, 3.6, 2.4, 2.9, 3.2]}

frame = pd.DataFrame(data)

frame 

,state,year,pop
0,Ohio,2000,1.5
1,Ohio,2001,1.7
2,Ohio,2002,3.6
3,Nevada,2001,2.4
4,Nevada,2002,2.9
5,Nevada,2003,3.2


For **large** DataFrames, the ``head`` method selects only the first five rows:

In [847]:
frame.head()

,state,year,pop
0,Ohio,2000,1.5
1,Ohio,2001,1.7
2,Ohio,2002,3.6
3,Nevada,2001,2.4
4,Nevada,2002,2.9


Similarly, ``tail`` returns the **last five** rows:

In [848]:
frame.tail()

,state,year,pop
1,Ohio,2001,1.7
2,Ohio,2002,3.6
3,Nevada,2001,2.4
4,Nevada,2002,2.9
5,Nevada,2003,3.2


If you specify a sequence of columns, the DataFrame's columns will be arranged in that order:

In [849]:
pd.DataFrame(data, columns = ["year", "state", "pop"])

,year,state,pop
0,2000,Ohio,1.5
1,2001,Ohio,1.7
2,2002,Ohio,3.6
3,2001,Nevada,2.4
4,2002,Nevada,2.9
5,2003,Nevada,3.2


If you pass a column that isn't contained in the dictionary, it will appear with missing values in the result:

In [850]:
frame2 = pd.DataFrame(data, columns=["year", "state", "pop", "debt"])

frame2

,year,state,pop,debt
0,2000,Ohio,1.5,NaN
1,2001,Ohio,1.7,NaN
2,2002,Ohio,3.6,NaN
3,2001,Nevada,2.4,NaN
4,2002,Nevada,2.9,NaN
5,2003,Nevada,3.2,NaN


A column in a DataFrame can be retrieved as a Series either by a dictionary-like notation or by using the dot attribute notation:

In [851]:
frame2["state"]

0      Ohio
1      Ohio
2      Ohio
3    Nevada
4    Nevada
5    Nevada
Name: state, dtype: str

In [852]:
frame2.year

0    2000
1    2001
2    2002
3    2001
4    2002
5    2003
Name: year, dtype: int64

Note that the returned Series have the same index as the DataFrame, and their ``name`` attribute has been appropriately set. 

Rows can also be retrieved by position or name with the special ``iloc`` and ``loc`` attributes. 

In [853]:
frame2.iloc[1]

year     2001
state    Ohio
pop       1.7
debt      NaN
Name: 1, dtype: object

In [854]:
frame2.iloc[2]

year     2002
state    Ohio
pop       3.6
debt      NaN
Name: 2, dtype: object

Columns can be modified by assignment. For example, the empty ``debt`` column could be assigned a scalar value or an array of values:

In [855]:
frame2["debt"] = 16.5

frame2

,year,state,pop,debt
0,2000,Ohio,1.5,16.5
1,2001,Ohio,1.7,16.5
2,2002,Ohio,3.6,16.5
3,2001,Nevada,2.4,16.5
4,2002,Nevada,2.9,16.5
5,2003,Nevada,3.2,16.5


In [856]:
frame2["debt"] = np.arange(6.)

frame2

,year,state,pop,debt
0,2000,Ohio,1.5,0.0
1,2001,Ohio,1.7,1.0
2,2002,Ohio,3.6,2.0
3,2001,Nevada,2.4,3.0
4,2002,Nevada,2.9,4.0
5,2003,Nevada,3.2,5.0


When you are assigning lists or arrays to a column, the value's length must match the length of the DataFrame. If you assign a Series, its labels will be realigned exactly to the DataFrame's index, inserting missing values in any index values not present:

In [857]:
val = pd.Series([-1.2, -1.5, -1.7], index = ["two", "four", "five"])

frame2["debt"] = val

frame2

,year,state,pop,debt
0,2000,Ohio,1.5,NaN
1,2001,Ohio,1.7,NaN
2,2002,Ohio,3.6,NaN
3,2001,Nevada,2.4,NaN
4,2002,Nevada,2.9,NaN
5,2003,Nevada,3.2,NaN


Assigning a column that doesn't exist will create a new column. 

The ``del`` keyword will delete columns with a dictionary. As an example, I first add a new column of Boolean values where the ``state`` column equals "Ohio":

In [858]:
frame2["eastern"] = frame2["state"] == "Ohio"

frame2

,year,state,pop,debt,eastern
0,2000,Ohio,1.5,NaN,True
1,2001,Ohio,1.7,NaN,True
2,2002,Ohio,3.6,NaN,True
3,2001,Nevada,2.4,NaN,False
4,2002,Nevada,2.9,NaN,False
5,2003,Nevada,3.2,NaN,False


The ``del`` method can then be used to remove this column"

In [859]:
del frame2["eastern"]

In [860]:
frame2.columns

Index(['year', 'state', 'pop', 'debt'], dtype='str')

Another common form of data is a nested dictionary of dictionaries:

In [861]:
populations = {"Ohio": {2000: 1.5, 2001: 1.7, 2002: 3.6}, 
            "Nevada":{2001:2.4, 2002:2.9}}

If the nested dictionary is passed to the DataFrame, pandas will interpret the outer dictionary keys as the columns, and the inner keys as the rows indices:

In [862]:
frame3 = pd.DataFrame(populations)

frame3

,Ohio,Nevada
2000,1.5,NaN
2001,1.7,2.4
2002,3.6,2.9


You can transpose the DataFrame (swap rows and columns) with similar syntax to a NumPy array:

In [863]:
frame3.T

,2000,2001,2002
Ohio,1.5,1.7,3.6
Nevada,NaN,2.4,2.9


> Note that transposing discards the column data types if the columns do not all have the same data type, so transposing and then transposing back may lose the previous type information. The columns become arrays of pure Python objects in this case. 

The keys in the inner dictionaries are combined to form the index in the result This isn't true if an explicit index is specified:

In [864]:
pd.DataFrame(populations, index = [2001, 2002, 2003])

,Ohio,Nevada
2001,1.7,2.4
2002,3.6,2.9
2003,NaN,NaN


Dictionaries of Series are treated in much the same way:

In [865]:
pdata = {"Ohio": frame3["Ohio"][:-1], 
        "Nevada": frame3["Nevada"][:2]}

In [866]:
pd.DataFrame(pdata)

,Ohio,Nevada
2000,1.5,NaN
2001,1.7,2.4


If a DataFrames ``index`` and ``columns`` have their ``name`` attributes set, these will also be displayed:

In [867]:
frame3.index.name = "year"

frame3.columns.name = "state"

frame3

state,Ohio,Nevada
year,,
2000,1.5,NaN
2001,1.7,2.4
2002,3.6,2.9


Unlike Series, DataFrame does not have a ``name`` attribute. DataFrame's ``to_numpy`` method returns the data contained in the DataFrame as a two-dimensional ndarray:

In [868]:
frame3.to_numpy

<bound method DataFrame.to_numpy of state  Ohio  Nevada
year               
2000    1.5     NaN
2001    1.7     2.4
2002    3.6     2.9>

If the DataFrame's columns are different data types, the data type of the returned array will be chosen to accomodate all of the columns:

In [869]:
frame2.to_numpy

<bound method DataFrame.to_numpy of    year   state  pop  debt
0  2000    Ohio  1.5   NaN
1  2001    Ohio  1.7   NaN
2  2002    Ohio  3.6   NaN
3  2001  Nevada  2.4   NaN
4  2002  Nevada  2.9   NaN
5  2003  Nevada  3.2   NaN>

## Index Objects 

pandas's Index objects are responsible for holding the axis labels (including a DataFrame's column name) and other metadata (like the axis name or names). Any array of other sequence of labels you use when constructing a Series or DataFrame is internally converted to an Index:

In [870]:
obj = pd.Series(np.arange(3), index = ["a", "b", "c"])

index = obj.index

index

Index(['a', 'b', 'c'], dtype='str')

In [871]:
index[1:]

Index(['b', 'c'], dtype='str')

Index objects are immutable and thus can't be modified by the user:

In [872]:
try:
    index[1] = "d"
except:
    "not valid"  

Immutability makes it safer to share Index objects among data structures:

In [873]:
labels = pd.Index(np.arange(3))

labels

Index([0, 1, 2], dtype='int64')

In [874]:
obj2 = pd.Series([1.5, -2.5, 0], index = labels)

obj2

0    1.5
1   -2.5
2    0.0
dtype: float64

In [875]:
obj2.index is labels

True

In addition to being array-like, an Index also behaves like a fixed-size set:

In [876]:
frame3

state,Ohio,Nevada
year,,
2000,1.5,NaN
2001,1.7,2.4
2002,3.6,2.9


In [877]:
frame3.columns

Index(['Ohio', 'Nevada'], dtype='str', name='state')

In [878]:
"Ohio" in frame3.columns

True

In [879]:
2003 in frame3.index

False

Unlike Python sets, a pandas Index can contain duplicate lables:

In [880]:
pd.Index(["foo", "foo","bar", "bar"])


Index(['foo', 'foo', 'bar', 'bar'], dtype='str')

Selections with duplicate labels will select all occurrences of that label. 

Each Index has a number of methods and properties for set logic, which answer other common questions about the data it contains. Some useful ones are summarized in Table 5-2:

## Essential Functionality

This section will walk you through the fundemental mechanics of interacting with the data contained in a Series or DataFrame. In the chapters to come, we will delve more deeply into data analysis and manipulation topics using pandas. 


## Reindexing

An important method on pandas objects is ``reindex``, which means to create a new object with the values rearranged to align with the new index. Consider an example:

In [881]:
obj = pd.Series([4.5, 7.2, -5.3, 3.6], index = ["d", "b", "a", "c"])

obj

d    4.5
b    7.2
a   -5.3
c    3.6
dtype: float64

Calling ``reindex`` on this Series rearranges the data according to the new index, introducing missing values if any index values were not already present:

In [882]:
obj2 = obj.reindex(["a", "b", "c", "d", "e" ])

obj2

a   -5.3
b    7.2
c    3.6
d    4.5
e    NaN
dtype: float64

For **ordered** data like time series, you way want to do some interpolation or filling of values when reindexing. The ``method`` option allows us to do this, using a method such as ``ffill``, which forward-fills the values:

In [883]:
obj3 = pd.Series(["blue", "purple", "yellow"], index = [0,2,4])

obj3

0      blue
2    purple
4    yellow
dtype: str

In [884]:
obj3.reindex(np.arange(6), method = "ffill")

0      blue
1      blue
2    purple
3    purple
4    yellow
5    yellow
dtype: str

With DataFrame, ``reindex`` can alter the (row) index, columns, or both. When passed only a sequence, it reindexes the rows in the result:

In [885]:
frame = pd.DataFrame(np.arange(9).reshape((3,3)), 
                    index = ["a", "c", "d"], 
                    columns=["Ohio", "Texas", "California"])

frame

,Ohio,Texas,California
a,0,1,2
c,3,4,5
d,6,7,8


In [886]:
frame2 = frame.reindex(index = ["a", "b", "c", "d"])

frame2

,Ohio,Texas,California
a,0.0,1.0,2.0
b,NaN,NaN,NaN
c,3.0,4.0,5.0
d,6.0,7.0,8.0


The columns can be reindexed with the ``columns`` keyword:

In [887]:
states = ["Texas", "Utah", "California"]

frame.reindex(columns=states)

,Texas,Utah,California
a,1,NaN,2
c,4,NaN,5
d,7,NaN,8


Because ``"Ohio"`` was not in ``states``, the data for that column is dropped from the result. 

Another way to reindex a particular axis is to pass the new axis labels as a positional argument and then specify the axis to reindex with the ``axis`` keyword:

In [888]:
frame.reindex(states, axis="columns")

,Texas,Utah,California
a,1,NaN,2
c,4,NaN,5
d,7,NaN,8


| Argument | Description |
|----------|----------|
| ``labels`` | New sequence to use as an index. Can be Index instance or any other sequence-like Python data struture.|
|``index``| Use the passed sequence as the new index labels |
|``columns``| Use the passed sequence as the new columns labels | 
|``axis``| The axis to reindex, whether "index" (rows) or "columns". The default is "index". You can alternately do ``reindex(index = new_labels)`` or ``reindex(columns = new_labels)``.|
|``method``| Interpolation (fill) method; ``"ffill"`` fills forward, while ``"bfill" fills backward. |
|``fill_value``| Substitute value to use when introducing missing data by reindexing. Use ``fill_value = "missing`` (the default behaviour) when you want absent labels to have null values in the result|
|``limit``| When forward filling or backfilling, the maximum size gap (in number of elements) to fill.|
|``tolerace``|When forward filling or backfilling, the maximum size gap (in absolute numeric distance) to fill for inexact matches|
|``level``| Match simple Index on level of MultiIndex; otherwise select subset of.|
|``copy``|If ``True``, always copy underlying data even if the new index is equivalent to the old index; if ``False``, do not copy the data when the indexes are equivalent|



You can also reindex by using the ``loc`` operator, and many users prefer to always do it this way. this works only if all of the new index labels already exist in the DataFrame (whereas ``reindex`` will insert missing data data for new labels):

In [889]:
frame.loc[["a", "d", "c"], ["California", "Texas"]]

,California,Texas
a,2,1
d,8,7
c,5,4


## Dropping Entries from an Axis 

Dropping one or more entries from an axis is simple if you already have an index array or list without those entries, since you can use the ``reindex`` method or ``.loc-`` based indexing. As that can require a bit of munging and set logic, the ``drop`` method will return a new object with the indicated value or values deleted from an axis:

In [890]:
obj = pd.Series(np.arange(5.), index = ["a", "b", "c", "d", "e"])

obj

a    0.0
b    1.0
c    2.0
d    3.0
e    4.0
dtype: float64

In [891]:
new_obj = obj.drop("c")

new_obj

a    0.0
b    1.0
d    3.0
e    4.0
dtype: float64

In [892]:
new_obj = obj.drop(["c", "d"])

new_obj

a    0.0
b    1.0
e    4.0
dtype: float64

With DataFrame, index values can be deleted from either axis. To illustrate this, we first create an example DataFrame:

In [893]:
data = pd.DataFrame(np.arange(16).reshape((4, 4)),
                index = ["Ohio", "Colorado", "Utah", "New York"],
                columns=["one", "two", "three", "four"])

data

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


Calling ``drop`` with a sequence of labels will drop values from the rows lables (axis = 0):

In [894]:
data.drop(index = ["Colorado", "Ohio"])

,one,two,three,four
Utah,8,9,10,11
New York,12,13,14,15


To drop labels from the columns, instead use the ``columns`` keyword:

In [895]:
data.drop(columns=["two"])

,one,three,four
Ohio,0,2,3
Colorado,4,6,7
Utah,8,10,11
New York,12,14,15


You can drop values from the columns by passing ``axis=1`` (which is like NumPy) or ``axis = "columns``:

In [896]:
data.drop("two", axis=1)

,one,three,four
Ohio,0,2,3
Colorado,4,6,7
Utah,8,10,11
New York,12,14,15


In [897]:
data.drop(["two", "four"], axis="columns")

,one,three
Ohio,0,2
Colorado,4,6
Utah,8,10
New York,12,14


## Indexing, Selection, and Filtering 

Series indexing (obj[...]) works analogously to NumPy array indexing, expcept you can use the Series's index values instead of only integers. Here are some examples of this:

In [898]:
obj = pd.Series(np.arange(4.), index = ["a", "b", "c", "d"])

obj

a    0.0
b    1.0
c    2.0
d    3.0
dtype: float64

In [899]:
obj["b"]

np.float64(1.0)

In [900]:
obj[2:4]

c    2.0
d    3.0
dtype: float64

In [901]:
obj[["b", "a", "d"]]

b    1.0
a    0.0
d    3.0
dtype: float64

In [902]:
obj[obj <2]

a    0.0
b    1.0
dtype: float64

The reason to prefer the ``loc`` operator is because of the different treatment of integers when indexing with []. Regular []-based will treat integers as labels if the index contains integers, so the behaviour differs depending on the data type of the index. For example:

In [903]:
obj1 = pd.Series([1, 2, 3], index = [2, 0, 1])

obj2 = pd.Series([1, 2, 3], index = ["a", "b", "c"])

obj1

2    1
0    2
1    3
dtype: int64

In [904]:
obj2

a    1
b    2
c    3
dtype: int64

In [905]:
obj1[[0,1,2]]

0    2
1    3
2    1
dtype: int64

In [906]:
obj2[["a", "b" , "c"]]

a    1
b    2
c    3
dtype: int64

Since ``loc`` operator indexes exclusively with labels, there is also an ``iloc`` operator that indexes exlusively with integers to work consistently whether or not the index contains integers:

In [907]:
obj1.iloc[[0, 1, 2]]

2    1
0    2
1    3
dtype: int64

In [908]:
obj2.iloc[[0,1,2]]

a    1
b    2
c    3
dtype: int64

You can also slice with labels, but it works differently from normal Python slicing in that the endpoint is inclusive:

In [909]:
obj2.loc["b":"c"]

b    2
c    3
dtype: int64

Assigning values using these methods modifies the corresponding section of the Series:

In [910]:
obj2.loc["b":"c"] = 5

obj2

a    1
b    5
c    5
dtype: int64

Indexing into a DataFrame retrieves one or more columns either with a single value or sequence:

In [911]:
data = pd.DataFrame(np.arange(16).reshape((4,4)),
                    index = ["Ohio", "Colorado", "Utah", "New York"],
                    columns = ["one", "two", "three", "four"])


data

,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


In [912]:
data["two"]

Ohio         1
Colorado     5
Utah         9
New York    13
Name: two, dtype: int64

In [913]:
data[["three", "one"]]

,three,one
Ohio,2,0
Colorado,6,4
Utah,10,8
New York,14,12


Indexing like this has a few special cases. The first is slicing or selecting data with a Boolean array:

In [914]:
data[:2]



,one,two,three,four
Ohio,0,1,2,3
Colorado,4,5,6,7


In [915]:
data[data["three"] > 5]

,one,two,three,four
Colorado,4,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


The row selection syntax ``data[:2]`` is provided as a convenience. Passing a single element or a list to the [] operator selects columns. 

Another use case is indexing with a Booelan DataFrame, such as one produced by a scalar comparison. Consider a DataFrame with all Boolean values produced by comparing with a scalar value;



In [916]:
data < 5

,one,two,three,four
Ohio,True,True,True,True
Colorado,True,False,False,False
Utah,False,False,False,False
New York,False,False,False,False


We can use this DataFrame to assign the value 0 to each location with the value ``True``, like so:

In [917]:
data[data < 5] = 0

data

,one,two,three,four
Ohio,0,0,0,0
Colorado,0,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


### Selection on DataFrame with loc and iloc

Like Series, DataFrame has special attributes ``loc`` and ``iloc`` for label-based and integer-based indexing, respectively. Since DataFrame is two-dimensional, you can select a subset of the rows and columns with NumPy-like notation using either axis labels (``loc``) or integers (``iloc``). 

As a first example, let's select a single row by label: 

In [918]:
data

,one,two,three,four
Ohio,0,0,0,0
Colorado,0,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


In [919]:
data.loc["Colorado"]

one      0
two      5
three    6
four     7
Name: Colorado, dtype: int64

The result of selecting a single row is a Series with an index that contains the DataFrame's column labels. To select multiple roles, creating a new DataFrame, pass a sequence of labels:

In [920]:
data.loc[["Colorado", "New York"]]

,one,two,three,four
Colorado,0,5,6,7
New York,12,13,14,15


You can combine both row and column selection in ``loc`` by seperating the selections with a comma:

In [921]:
data.loc["Colorado", ["two", "three"]]

two      5
three    6
Name: Colorado, dtype: int64

In [922]:
data.iloc[2]

one       8
two       9
three    10
four     11
Name: Utah, dtype: int64

In [923]:
data.iloc[[2, 1]]

,one,two,three,four
Utah,8,9,10,11
Colorado,0,5,6,7


In [924]:
data.iloc[:, :3][data.three > 5]

,one,two,three
Colorado,0,5,6
Utah,8,9,10
New York,12,13,14


Boolean arrays can be used with ``loc`` but not ``iloc``:

In [925]:
data.loc[data.three >= 2]

,one,two,three,four
Colorado,0,5,6,7
Utah,8,9,10,11
New York,12,13,14,15


## Indexing options with DataFrame

| Type | Notes |
|------|-------|
| `df[column]` | Select single column or sequence of columns from the DataFrame; special case conveniences: Boolean array (filter rows), slice (slice rows), or Boolean DataFrame (set values based on some criterion) |
| `df.loc[rows]` | Select single row or subset of rows from the DataFrame by label |
| `df.loc[:, cols]` | Select single column or subset of columns by label |
| `df.loc[rows, cols]` | Select both row(s) and column(s) by label |
| `df.iloc[rows]` | Select single row or subset of rows from the DataFrame by integer position |
| `df.iloc[:, cols]` | Select single column or subset of columns by integer position |
| `df.iloc[rows, cols]` | Select both row(s) and column(s) by integer position |
| `df.at[row, col]` | Select a single scalar value by row and column label |
| `df.iat[row, col]` | Select a single scalar value by row and column integer position |
| `reindex` method | Select either rows or columns by label |

### Integer indexing pitfalls 

Working with pandas objects indexed by integers can be a stumbling block for new users since they work differently from built-in Python data structures like lists and tuples. For example, you might not expect the following code to generate an error:

In [926]:
ser = pd.Series(np.arange(3.))

ser

0    0.0
1    1.0
2    2.0
dtype: float64

In [927]:
try:
    ser[-1]
except:
    "doesn't work"

In this case, pandas could "fall back" on integer indexing, but it is difficult to do this in general without introducing subtle bugs into the user code. Here we have an index containing ``0``, ``1``, and ``2``, but pandas does not want to guess what the user wants (label-based indexing or position-based):

In [928]:
ser

0    0.0
1    1.0
2    2.0
dtype: float64

On the other hand, with a noninteger index, there is no such ambiguity:

In [929]:
ser2 = pd.Series(np.arange(3.), index= ["a", "b", "c"])

ser2.iloc[-1]

np.float64(2.0)

If you have an axis index containing integers, data selection will always be label oriented. 

On the other hand integer, slicing with integers is always integer oriented. 

In [930]:
ser[:2]

0    0.0
1    1.0
dtype: float64

### Pitfalls with chained indexing 

In the previous section we looked at how you can do flexible selections on a DataFrame using ``loc`` and ``iloc``. These indexing attributes can also be used to modify DataFrame objects in place, but doing so requires some care:

For example, in the example DataFrame above, we can assign to a column or row by label or integer position:

In [931]:
data.loc[:, "one"] = 1

data

,one,two,three,four
Ohio,1,0,0,0
Colorado,1,5,6,7
Utah,1,9,10,11
New York,1,13,14,15


In [932]:
data.iloc[2] = 5

data

,one,two,three,four
Ohio,1,0,0,0
Colorado,1,5,6,7
Utah,5,5,5,5
New York,1,13,14,15


In [933]:
data.loc[data["four"] > 5] = 3

data

,one,two,three,four
Ohio,1,0,0,0
Colorado,3,3,3,3
Utah,5,5,5,5
New York,3,3,3,3


A common gotcha for new pandas users is to chain selection when assigning, like this:


In [934]:
data.loc[data.three == 5]["three"] = 6

/var/folders/72/xf7wbb39361gc9hk7n78n1j00000gn/T/ipykernel_82486/867481848.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  data.loc[data.three == 5]["three"] = 6


Instead

In [935]:
data.loc[data.three == 5, "three"]= 6

data

,one,two,three,four
Ohio,1,0,0,0
Colorado,3,3,3,3
Utah,5,5,6,5
New York,3,3,3,3


## Function Application and Mapping

NumPy ufuncs (element-wise array methods) also work with pandas objects:

In [936]:
frame = pd.DataFrame(np.random.standard_normal((4, 3)),
                    columns = list("bde"),
                    index = ["Utah", "Ohio", "Texas", "Oregon"])

frame

,b,d,e
Utah,1.546093,0.559827,-0.380147
Ohio,-0.423859,0.241252,0.389825
Texas,-1.163123,-0.971845,1.098014
Oregon,2.073829,-0.294546,2.498390


In [937]:
np.abs(frame)

,b,d,e
Utah,1.546093,0.559827,0.380147
Ohio,0.423859,0.241252,0.389825
Texas,1.163123,0.971845,1.098014
Oregon,2.073829,0.294546,2.498390


Another frequent operation is applying a function on one-dimensional arrays to each column or row. DataFrame's ``aaply`` method does exactly this:

In [938]:
def f1(x):
    return x.max() - x.min()

frame.apply(f1)



b    3.236952
d    1.531672
e    2.878536
dtype: float64

Here the function ``f``, which computes the difference between the maximum and minimum of a Series, is invoked once on each column in ``frame``. The result is a Series having the columns of ``frame`` as its index:

If you pass ``axis = "column"`` to ``apply``, the function will be invoked once per row instead. A helpful way to think about this as "apply accross the columns":

In [939]:
frame.apply(f1, axis="columns")

Utah      1.926240
Ohio      0.813684
Texas     2.261137
Oregon    2.792935
dtype: float64

Many of the most common array statistics (like ``sum`` and ``mean`` ) are DataFrame methods, so using ``apply`` is not neccessary. 

The function passed to ``apply`` need to return a scalar value; it can also return a Series with multiple values:

In [940]:
def f2(x):
    return pd.Series([x.min(), x.max()], index = ["min", "max"])

frame.apply(f2)

,b,d,e
min,-1.163123,-0.971845,-0.380147
max,2.073829,0.559827,2.498390


Element-wise Python functions can be used, too. Suppose you wanted to compute a formatted string from each floating-point value in ``frame``. You can do this with ``applymap``:



In [941]:
def my_format(x):
    return f"{x:.2f}"

frame.map(my_format)

,b,d,e
Utah,1.55,0.56,-0.38
Ohio,-0.42,0.24,0.39
Texas,-1.16,-0.97,1.10
Oregon,2.07,-0.29,2.50


## Sorting and Ranking

Sorting a dataset by some criterion is another important built-in operation. To sort lexicographically by row or column label, use the ``sort_index`` method, which returns a new, sorted object:


In [942]:
obj = pd.Series(np.arange(4), index = ["d", "a", "b", "c"])

obj

d    0
a    1
b    2
c    3
dtype: int64

In [943]:
obj.sort_index()

a    1
b    2
c    3
d    0
dtype: int64

With a DataFrame, you can sort by index on either axis:

In [944]:
frame = pd.DataFrame(np.arange(8).reshape((2,4)), 
                    index = ["three", "one"], 
                    columns=["d", "a", "b", "c"])

frame

,d,a,b,c
three,0,1,2,3
one,4,5,6,7


In [945]:
frame.sort_index()

,d,a,b,c
one,4,5,6,7
three,0,1,2,3


In [946]:
frame.sort_index(axis = "columns")

,a,b,c,d
three,1,2,3,0
one,5,6,7,4


The data is sorted in **ascending** order by default byt can be sorted in **decreasing** order, too:

In [947]:
frame.sort_index(axis="columns", ascending=False)

,d,c,b,a
three,0,3,2,1
one,4,7,6,5


To sort a Series by its values, use its ``sort_values`` method:

In [948]:
obj = pd.Series([4, 7, -3, 2])

obj.sort_values()


2   -3
3    2
0    4
1    7
dtype: int64

Any missing values are sorted to the end of the Series by default:

In [949]:
obj = pd.Series([4, np.nan, 7, np.nan, -3.2])

obj.sort_values()

4   -3.2
0    4.0
2    7.0
1    NaN
3    NaN
dtype: float64

However they can be sorted to the start instead using the ``na_position`` option:

In [951]:
obj = obj.sort_values(na_position="first")
print(obj)

1    NaN
3    NaN
4   -3.2
0    4.0
2    7.0
dtype: float64
